# BirdCLEF+ 2025 - Pretrained models, A3: Bird-MAE-Base + MLP head, inference (submission notebook)

**CPU only, internet off.** This notebook uses the same protocol as `infer-v2`: twelve 5 s windows per 1-minute soundscape, with the same `row_id` and columns. Four worker processes each handle whole files, with one thread each.

It needs this competition's data, the Bird-MAE folder from the private dataset `strippedbanana/sc4000-birdclef-m3-birdmae-base`, the MLP head from `sc4000-birdclef-m3-a3-distill`, and the offline wheels from the A3 embed notebook. When `test_soundscapes/` is empty, it does a dry run on training soundscapes and projects 700 files against the 75 minute budget.

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"                # set before numpy loads; each worker uses one core

import glob, hashlib, json, subprocess, sys, time
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd

T_NOTEBOOK = time.time()
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"     # local test only, never set on Kaggle
PINS = {"transformers": "4.38.0", "tokenizers": "0.15.2", "huggingface_hub": "0.21.4"}
PYDEPS = Path(os.environ.get("M3_PYDEPS", "/tmp/bmae_pydeps"))
NUM_WORKERS = int(os.environ.get("M3_INFER_WORKERS", "4"))      # 0 = no Pool (Windows tests)
DRY_FILES = int(os.environ.get("M3_DRY_FILES", "20"))
BUDGET_MIN = 75

def find_parent(marker, env, fallback):
    if os.environ.get(env):
        return Path(os.environ[env])
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    return Path(hits[0]).parent if hits else Path(fallback)

DATA = find_parent("birdclef-2025/sample_submission.csv", "M3_DATA_DIR", "../../data")
HEAD_NAME = "birdmae_head_distill_coral.npz"          # exact file from a3-distill, never the embed kernel's birdmae_head.npz
HEAD_DIR = find_parent(HEAD_NAME, "M3_MODEL_DIR", "../a3_distill")
if not (HEAD_DIR / HEAD_NAME).is_file():
    raise FileNotFoundError(f"{HEAD_NAME} not found in {HEAD_DIR}: attach strippedbanana/sc4000-birdclef-m3-a3-distill")
WHEELS_DIR = find_parent("bmae_wheels", "M3_WHEELS_DIR", "../a3_birdmae_embed")     # the embed notebook's output
MODEL_DIR = find_parent("modeling_bird_mae.py", "M3_BIRDMAE_DIR", ".")
assert (MODEL_DIR / "model.safetensors").exists(), \
    f"Bird-MAE not found: attach strippedbanana/sc4000-birdclef-m3-birdmae-base ({MODEL_DIR})"

# Pinned packages go to a private folder from the wheels saved by the embed notebook, so the image's
# own packages stay as they are. Nothing imports torch or transformers in this process; the workers do.
if not SMOKE_TEST:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--target", str(PYDEPS),
                        "--find-links", str(WHEELS_DIR / "bmae_wheels"), *[f"{k}=={v}" for k, v in PINS.items()]],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Could not install the pinned wheels: {r.stderr[-400:]}")

head_cfg = json.load(open(HEAD_DIR / "head_cfg.json"))
assert head_cfg["best_head"] == "mlp", f"expected the MLP head, got {head_cfg['best_head']}"
species = json.load(open(HEAD_DIR / "species.json"))
sample_sub = pd.read_csv(DATA / "sample_submission.csv")
assert list(sample_sub.columns[1:]) == species, "species order differs from sample_submission"
print("Data:", DATA, "| head:", HEAD_DIR, "| wheels:", WHEELS_DIR, "| model:", MODEL_DIR, "| CPUs:", os.cpu_count(), "| workers:", NUM_WORKERS)
print("Head:", head_cfg["best_head"], "| HF revision:", head_cfg["hf_revision"], "| pins:", PINS)

In [ ]:
# The attached folder must be byte-identical to what the embeddings were made with.
for name, want in head_cfg["birdmae_files_sha256"].items():
    h = hashlib.sha256()
    with open(MODEL_DIR / name, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    if h.hexdigest() != want:
        raise RuntimeError(f"{name} differs from the file used for the embeddings: {h.hexdigest()}")

test_files = sorted((DATA / "test_soundscapes").glob("*.ogg"))
DRY_RUN = len(test_files) == 0
if DRY_RUN:
    test_files = sorted((DATA / "train_soundscapes").glob("*.ogg"))[:DRY_FILES]
print(f"{'DRY RUN on' if DRY_RUN else 'Predicting'} {len(test_files)} soundscapes")

## Worker code (one model per process, one thread each)

In [ ]:
import os, sys, tempfile, time
import numpy as np
import soundfile as sf
from pathlib import Path
from scipy.signal import resample_poly

SR, WIN_SEC = 32_000, 5
WIN = SR * WIN_SEC          # 160000 samples, Bird-MAE's native clip length

def to_32k_mono(y, sr):
    """Shared with the other A3 notebook, keep both copies identical."""
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != SR:
        y = resample_poly(y, SR, sr)
    return y.astype(np.float32)

def head_predict(head, emb):
    """Sigmoid probabilities from saved head arrays. Same as A2; shared with the other A3 notebook."""
    x = (emb - head["mean"]) / head["std"]
    n_layers = int(head["n_layers"])
    for i in range(n_layers):
        x = x @ head[f"W{i}"].T + head[f"b{i}"]
        if i < n_layers - 1:
            x = np.maximum(x, 0.0)       # ReLU
    return 1.0 / (1.0 + np.exp(-x))

def check_head(head, torch):
    """Hidden linear, ReLU, final linear in torch (dropout off) must match head_predict on random input."""
    assert int(head["n_layers"]) == 2, f"expected an MLP head with 2 linear layers, got {int(head['n_layers'])}"
    W0, W1 = head["W0"], head["W1"]
    net = torch.nn.Sequential(torch.nn.Linear(W0.shape[1], W0.shape[0]), torch.nn.ReLU(), torch.nn.Dropout(0.2),
                              torch.nn.Linear(W1.shape[1], W1.shape[0])).eval()
    net.load_state_dict({"0.weight": torch.from_numpy(W0), "0.bias": torch.from_numpy(head["b0"]),
                         "3.weight": torch.from_numpy(W1), "3.bias": torch.from_numpy(head["b1"])})
    z = np.random.default_rng(0).standard_normal((32, W0.shape[1])).astype(np.float32)
    with torch.no_grad():
        p_torch = torch.sigmoid(net(torch.from_numpy(z))).numpy()
    diff = float(np.abs(head_predict(head, head["mean"] + head["std"] * z) - p_torch).max())
    assert diff < 1e-4, f"numpy head differs from torch head: {diff}"
    return diff

_torch, _fe, _model, _head, _init_s = None, None, None, None, 0.0

def init_worker(model_dir, head_path, pydeps):
    global _torch, _fe, _model, _head, _init_s
    t0 = time.perf_counter()
    # Own cache per process: transformers copies the repo's code there, and workers must not race on it.
    cache = os.path.join(tempfile.gettempdir(), f"hf_{os.getpid()}")
    os.environ.update(HF_HOME=cache, HF_MODULES_CACHE=os.path.join(cache, "modules"),
                      HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    if str(pydeps) not in sys.path:
        sys.path.insert(0, str(pydeps))
    import torch                       # first load happens inside the worker
    torch.set_num_threads(1)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:               # already set in this process (only in the no-Pool test)
        pass
    import transformers
    assert Path(transformers.__file__).is_relative_to(pydeps), "pinned transformers is not the one imported"
    from transformers import AutoFeatureExtractor, AutoModel
    _fe = AutoFeatureExtractor.from_pretrained(str(model_dir), trust_remote_code=True)
    _model = AutoModel.from_pretrained(str(model_dir), trust_remote_code=True).eval().requires_grad_(False)
    _head = dict(np.load(head_path))
    print(f"head check, numpy vs torch on random input: max abs diff {check_head(_head, torch):.2e}", flush=True)
    _torch = torch
    _init_s = time.perf_counter() - t0

def windows_to_specs(w):
    """One feature-extractor call per window, never a batched call: the repo pads the ~14 missing frames
    with the minimum of whatever batch it is given, so a batched call would make a window's input depend
    on its neighbours and differ between training and inference."""
    return _torch.cat([_fe(x) for x in w]).float()      # [n, 1, 512, 128]

def predict_file(path):
    t0 = time.perf_counter()
    y, sr = sf.read(str(path), dtype="float32")
    y = to_32k_mono(y, sr)
    t1 = time.perf_counter()
    n_win = max(1, int(np.ceil(len(y) / WIN)))
    y = np.pad(y, (0, n_win * WIN - len(y)))
    x = windows_to_specs(np.stack([y[i * WIN:(i + 1) * WIN] for i in range(n_win)]))
    t2 = time.perf_counter()
    with _torch.inference_mode():
        emb = _model(x)[0].numpy()     # the 12 windows of a file are one batch
    t3 = time.perf_counter()
    p = head_predict(_head, emb).astype(np.float32)
    t4 = time.perf_counter()
    # row_id = <file stem>_<end time in seconds>, e.g. soundscape_8358733_5
    row_ids = [f"{Path(path).stem}_{(i + 1) * WIN_SEC}" for i in range(n_win)]
    return row_ids, p, dict(decode=t1 - t0, mel=t2 - t1, model=t3 - t2, head=t4 - t3,
                            init=_init_s, pid=os.getpid())

## Predict, write `submission.csv`, report timing

In [ ]:
t0 = time.time()
initargs = (MODEL_DIR, HEAD_DIR / HEAD_NAME, PYDEPS)
if NUM_WORKERS > 0:
    with Pool(NUM_WORKERS, initializer=init_worker, initargs=initargs) as pool:
        results = list(pool.imap(predict_file, test_files, chunksize=1))
else:
    init_worker(*initargs)
    results = [predict_file(p) for p in test_files]
wall = time.time() - t0

row_ids = [r for ids, _, _ in results for r in ids]
preds = np.concatenate([p for _, p, _ in results])
sub = pd.DataFrame(preds, columns=species)
sub.insert(0, "row_id", row_ids)
sub.to_csv("submission.csv", index=False)
print(sub.shape); print(sub.iloc[:3, :4])

stage = lambda k: sum(r[2][k] for r in results)
inits = {r[2]["pid"]: r[2]["init"] for r in results}
init_wall = max(inits.values())
per_file = max(wall - init_wall, 0.0) / len(test_files)
projected = (init_wall + per_file * 700) / 60
total = time.time() - T_NOTEBOOK
print(f"TIMING | decode {stage('decode'):.1f}s | mel {stage('mel'):.1f}s | "
      f"model {stage('model'):.1f}s | head {stage('head'):.2f}s (summed over {len(inits)} workers)")
print(f"TIMING | model load per worker {init_wall:.1f}s | predict wall {wall:.1f}s | {len(test_files)} files | "
      f"{per_file:.2f}s per file")
print(f"TIMING | projected 700 files: {projected:.1f} min (budget {BUDGET_MIN}) -> "
      f"{'OK' if projected <= BUDGET_MIN else 'OVER BUDGET'}")
print(f"TOTAL notebook time: {total / 60:.2f} min")